# Homework 3 Notebook 
Author: Eugenia Tate 

Course: AIPI 530 by Dr. Natalia Summerville 

## Problem 1 — Campus courier tour (TSP)

A courier starts at office **O**, visits locations **A, B, and C** exactly once, and returns to O while minimizing the total travel distance.

#### Loading packages

In [1]:
import pandas as pd
from pyomo.environ import *
from pyomo.opt import SolverFactory

#### Creating Data
Locations: O, A, B, and C. Office O is the starting and ending point.
The distances are the same in both directions. The model uses directed arcs to represent travel between locations.

In [2]:
O, A, B, C = "O", "A", "B", "C"
DEPOT = O

distance_data = pd.DataFrame(columns=["From", "To", "Distance"])

distance_data["From"] = [O, O, O, O, A, A, A, A, B, B, B, B, C, C, C, C]

distance_data["To"] = [O, A, B, C, O, A, B, C, O, A, B, C, O, A, B, C]

distance_data["Distance"] = [0, 7, 12, 9, 7, 0, 6, 8, 12, 6, 0, 5, 9, 8, 5, 0]

distance_data

,From,To,Distance
0,O,O,0
1,O,A,7
2,O,B,12
3,O,C,9
4,A,O,7
5,A,A,0
6,A,B,6
7,A,C,8
8,B,O,12
9,B,A,6


#### Define Model

In [3]:
# create empty model
m = AbstractModel()

**Sets and parameters**

- $V = \{O, A, B, C\}$: set of campus locations
- $A = \{(i,j) \in V \times V : i \ne j\}$: set of arcs (no self-loops)
- $d_{ij}$: distance from location $i$ to location $j$ in miles
- $n = |V| = 4$: total number of locations

In [4]:
# define sets
m.V = Set()
m.A = Set(
    within=m.V * m.V, initialize=lambda m: [(i, j) for i in m.V for j in m.V if i != j]
)

# define parameters
m.d = Param(m.V, m.V)
m.n = Param(initialize=lambda m: len(m.V))

**Decision variables**

In [5]:
m.x = Var(m.A, domain=Binary)

m.u = Var(
    m.V,
    domain=NonNegativeIntegers,
    bounds=lambda m, i: (0, 0) if i == DEPOT else (1, m.n - 1),
)

#### Constraints

**Leave each location exactly once**

In [6]:
def Leave_Once_Rule(m, i):
    return sum(m.x[i, j] for j in m.V if j != i) == 1

m.LeaveOnce = Constraint(m.V, rule=Leave_Once_Rule)

**Enter each location exactly once**

In [7]:
def Enter_Once_Rule(m, j):
    return sum(m.x[i, j] for i in m.V if i != j) == 1

m.EnterOnce = Constraint(m.V, rule=Enter_Once_Rule)

**Subtour elimination (MTZ)**

The MTZ constraints prevent separate loops by requiring the visit order to increase when the courier travels directly between two non-depot locations.

In [8]:
def MTZ_Rule(m, i, j):
    if i == DEPOT or j == DEPOT:
        return Constraint.Skip

    return m.u[i] - m.u[j] + (m.n - 1) * m.x[i, j] <= m.n - 2

m.MTZ = Constraint(m.A, rule=MTZ_Rule)

**Objective: Minimize total distance**

In [9]:
def Min_Distance_Rule(m):
    return sum(m.d[i, j] * m.x[i, j] for (i, j) in m.A)

m.Min_Distance = Objective(rule=Min_Distance_Rule, sense=minimize)

#### Create Instance

In [10]:
instanceData = {
    None: {
        "V": {None: list(distance_data["From"].unique())},
        "d": distance_data.set_index(["From", "To"])["Distance"].to_dict(),
    }
}

instance = m.create_instance(instanceData)

#### Solve

In [11]:
# solve instance
solver = SolverFactory("glpk")
solution = solver.solve(instance)

print("Solver Status:")
print(solution["Solver"])

print("Optimum Objective Function Value:")
print(value(instance.Min_Distance))

Solver Status:

- Status: ok
  Termination condition: optimal
  Statistics: 
    Branch and bound: 
      Number of bounded subproblems: 5
      Number of created subproblems: 5
  Error rc: 0
  Time: 0.007111072540283203

Optimum Objective Function Value:
27.0


#### Print solution

In [12]:
# create output data: follow the arcs from the depot to get the tour in order
next_city = {i: j for (i, j) in instance.A if value(instance.x[i, j]) > 0.5}

tour = [DEPOT]
while next_city[tour[-1]] != DEPOT:
    tour.append(next_city[tour[-1]])
tour.append(DEPOT)

output_route = pd.DataFrame(
    [
        {"Stop": k + 1, "From": i, "To": j, "Distance": value(instance.d[i, j])}
        for k, (i, j) in enumerate(zip(tour[:-1], tour[1:]))
    ]
)

print("===== OPTIMAL TOUR =====")
print(" → ".join(tour))
print(f"Total Distance: {value(instance.Min_Distance):.0f}")

print("\nSelected decision variables:")
for i, j in instance.A:
    if value(instance.x[i, j]) > 0.5:
        print(f"x[{i},{j}] = {value(instance.x[i,j]):.0f}")

# print ordering variables, excluding the depot
print("\nOrdering variables:")
for i in instance.V:
    if i != DEPOT:
        print(f"u[{i}] = {value(instance.u[i]):.0f}")

output_route

===== OPTIMAL TOUR =====
O → A → B → C → O
Total Distance: 27

Selected decision variables:
x[O,A] = 1
x[A,B] = 1
x[B,C] = 1
x[C,O] = 1

Ordering variables:
u[A] = 1
u[B] = 2
u[C] = 3


,Stop,From,To,Distance
0,1,O,A,7
1,2,A,B,6
2,3,B,C,5
3,4,C,O,9


#### Explanation of Results

The optimal route is O → A → B → C → O, with a total travel distance of 27 miles. The reverse route is also optimal since the distances are same.

The route visits each campus location exactly once and returns to the dispatch office. The arrival and departure constraints are satisfied, and the MTZ constraints prevent disconnected subtours.

The solver reports an optimal solution - it is not just a local optimum - that is 27 miles is the minimum possible distance.

## Problem 2 - Campus Collection Routes (CVRP)

Two identical trucks start at office O and collect boxes from locations A, B, and C. Each location must be visited exactly once, and both trucks must return to O.

Each truck has a capacity of 10 boxes. The goal is to minimize the total distance traveled by both trucks.

In [13]:
# define packages
import pandas as pd
from pyomo.environ import *
from pyomo.opt import SolverFactory

#### Creating Data

The model uses the same campus locations and distances as Problem 1. Pickup quantities and truck capacity are added for this problem.

In [14]:
# distance data
O, A, B, C = "O", "A", "B", "C"
DEPOT = O

distance_data = pd.DataFrame(columns=["From", "To", "Distance"])
distance_data["From"] = [O, O, O, O, A, A, A, A, B, B, B, B, C, C, C, C]
distance_data["To"] = [O, A, B, C, O, A, B, C, O, A, B, C, O, A, B, C]
distance_data["Distance"] = [0, 7, 12, 9, 7, 0, 6, 8, 12, 6, 0, 5, 9, 8, 5, 0]
distance_data

,From,To,Distance
0,O,O,0
1,O,A,7
2,O,B,12
3,O,C,9
4,A,O,7
5,A,A,0
6,A,B,6
7,A,C,8
8,B,O,12
9,B,A,6


In [15]:
# pickup and fleet data
demand_data = pd.DataFrame(columns=["Customer", "Pickup"])

demand_data["Customer"] = [A, B, C]
demand_data["Pickup"] = [8, 6, 4]

TRUCKS = 2
CAPACITY = 10

demand_data

,Customer,Pickup
0,A,8
1,B,6
2,C,4


#### Defining Model

In [16]:
# create empty model
m = AbstractModel()

**Sets and Parameters**
- $V$: set of locations (depot and customers)
- $C = V \setminus \{O\}$: set of customers
- $A = \{(i,j) \in V \times V : i \ne j\}$: set of arcs (no self-loops)
- $c_{ij}$: distance from location $i$ to location $j$
- $d_i$: quantity picked up at customer $i$
- $K$: number of trucks
- $Q$: truck capacity

In [17]:
# define sets
m.V = Set()

m.C = Set(within=m.V, initialize=lambda m: [i for i in m.V if i != DEPOT])

m.A = Set(
    within=m.V * m.V, initialize=lambda m: [(i, j) for i in m.V for j in m.V if i != j]
)

# define parameters
m.c = Param(m.V, m.V)
m.d = Param(m.C)
m.K = Param()
m.Q = Param()

**Decision Variables**

In [18]:
# define decision variables
m.x = Var(m.A, domain=Binary)
m.u = Var(m.C, domain=NonNegativeReals, bounds=lambda m, i: (m.d[i], m.Q))

#### Constraints

Each customer must have one arrival

In [19]:
def Enter_Once_Rule(m, j):
    return sum(m.x[i, j] for i in m.V if i != j) == 1

m.EnterOnce = Constraint(m.C, rule=Enter_Once_Rule)

Each customer must have one departure

In [20]:
def Leave_Once_Rule(m, i):
    return sum(m.x[i, j] for j in m.V if j != i) == 1

m.LeaveOnce = Constraint(m.C, rule=Leave_Once_Rule)

Two trucks leave the depot

In [21]:
def Depot_Departures_Rule(m):
    return sum(m.x[DEPOT, j] for j in m.C) == m.K

m.DepotDepartures = Constraint(rule=Depot_Departures_Rule)

Two trucks return to the depot

In [22]:
def Depot_Returns_Rule(m):
    return sum(m.x[i, DEPOT] for i in m.C) == m.K

m.DepotReturns = Constraint(rule=Depot_Returns_Rule)

Load propagation: enforce truck capacity and prevent subtours

In [23]:
def Load_Rule(m, i, j):
    if i == DEPOT or j == DEPOT:
        return Constraint.Skip

    return m.u[i] - m.u[j] + m.Q * m.x[i, j] <= m.Q - m.d[j]

m.Load = Constraint(m.A, rule=Load_Rule)

**Objective:** minimize total dist

In [24]:
def Min_Distance_Rule(m):
    return sum(m.c[i, j] * m.x[i, j] for (i, j) in m.A)


m.Min_Distance = Objective(rule=Min_Distance_Rule, sense=minimize)

#### Create the Instance


In [25]:
# build and read instance data
instanceData = {
    None: {
        "V": {None: list(distance_data["From"].unique())},
        "c": distance_data.set_index(["From", "To"])["Distance"].to_dict(),
        "d": demand_data.set_index("Customer")["Pickup"].to_dict(),
        "K": {None: TRUCKS},
        "Q": {None: CAPACITY},
    }
}

instance = m.create_instance(instanceData)

#### Solve

In [26]:
# solve instance
solver = SolverFactory("glpk")
solution = solver.solve(instance)

print("Solver Status:")
print(solution["Solver"])

print("Optimum Objective Function Value:")
print(value(instance.Min_Distance))

Solver Status:

- Status: ok
  Termination condition: optimal
  Statistics: 
    Branch and bound: 
      Number of bounded subproblems: 3
      Number of created subproblems: 3
  Error rc: 0
  Time: 0.006621837615966797

Optimum Objective Function Value:
40.0


#### Print solution

In [27]:
# follow the selected arcs to build both routes
next_city = {
    i: j for (i, j) in instance.A if i != DEPOT and value(instance.x[i, j]) > 0.5
}

first_stops = [j for j in instance.C if value(instance.x[DEPOT, j]) > 0.5]

rows = []

print("===== OPTIMAL ROUTES =====")

for truck, first in enumerate(first_stops, start=1):
    route = [DEPOT, first]

    while route[-1] != DEPOT:
        route.append(next_city[route[-1]])

    load = 0

    for k, (i, j) in enumerate(zip(route[:-1], route[1:])):
        load += value(instance.d[j]) if j != DEPOT else 0

        rows.append(
            {
                "Truck": truck,
                "Stop": k + 1,
                "From": i,
                "To": j,
                "Distance": value(instance.c[i, j]),
                "Load": load,
            }
        )

    route_distance = sum(value(instance.c[i, j]) for i, j in zip(route[:-1], route[1:]))

    print(
        f"Truck {truck}: "
        + " -> ".join(route)
        + f"   Load: {load}   Distance: {route_distance}"
    )

print(f"\nTotal Distance: {value(instance.Min_Distance):.0f}")

output_routes = pd.DataFrame(rows)
output_routes

===== OPTIMAL ROUTES =====
Truck 1: O -> A -> O   Load: 8   Distance: 14
Truck 2: O -> C -> B -> O   Load: 10   Distance: 26

Total Distance: 40


,Truck,Stop,From,To,Distance,Load
0,1,1,O,A,7,8
1,1,2,A,O,7,8
2,2,1,O,C,9,4
3,2,2,C,B,5,10
4,2,3,B,O,12,10


In [28]:
# print selected routing variables
print("Selected routing variables:")
for i, j in instance.A:
    if value(instance.x[i, j]) > 0.5:
        print(f"x[{i},{j}] = {value(instance.x[i,j]):.0f}")

# print load variables
print("\nLoad variables:")
for i in instance.C:
    print(f"u[{i}] = {value(instance.u[i]):.2f}")

Selected routing variables:
x[O,A] = 1
x[O,C] = 1
x[A,O] = 1
x[B,O] = 1
x[C,B] = 1

Load variables:
u[A] = 8.00
u[B] = 10.00
u[C] = 4.00


### Explanation of Results

The optimal solution uses two trucks with a minimum combined travel distance of **40 miles**.

One truck travels O → A → O, collecting 8 boxes over 14 miles. The other travels O → B → C → O, collecting 10 boxes over 26 miles.

All three locations are visited exactly once, and both trucks return to the depot. The second truck reaches its full capacity of 10 boxes, while the first truck has 2 boxes of unused capacity.

The solver reports an optimal solution, confirming that 40 miles is the minimum total distance.